# Milestone 2 slice 3 — Encoder argument extraction (DeBERTa-v3-large)

Frame-conditioned **BIO role labeling**. Input:
`{frame} : … <t> {trigger} </t> …`; the model tags each sentence token with
a frame-element role (or O). At inference the label logits are **masked to the
frame's FEs** so only valid roles are emitted, and spans are scored with the
upstream weighting (non-core FEs count 0.5).

**Baseline to beat (argument extraction):** F1 `0.753` (our reproduction) /
`0.75` reported — this is the baseline's **weakest** task and our biggest
opportunity.

> `Runtime → GPU` (A100), fresh runtime, `Run all`.

In [ ]:
!nvidia-smi

In [ ]:
# --- Clone the private project repo + install pinned env --------------------
import os
try:
    from google.colab import userdata
    os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN") or ""
except Exception:
    os.environ.setdefault("GH_TOKEN", "")

![ -d Texture_Frames ] || git clone -q https://$GH_TOKEN@github.com/texturejc/Texture_Frames.git
!cd Texture_Frames && git fetch -q origin && git reset --hard -q origin/main && echo "project at $(git rev-parse --short HEAD)"
!pip install -q -r Texture_Frames/requirements-colab.txt

In [ ]:
# numpy MUST be 2.x. If 1.x: DELETE runtime (not Restart), then Run all.
import numpy, numpy.strings, torch, transformers
print("numpy", numpy.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")

import nltk
for pkg in ["framenet_v17", "wordnet", "omw-1.4"]:
    nltk.download(pkg)

## Sanity check — input + BIO labels

Confirm the combined input and the FE spans line up before training.

In [ ]:
import args_data
from lexicon import Lexicon

lex = Lexicon()
print("FE vocab size:", len(lex.fe_vocab()))

examples = args_data.load_args_examples("dev")
print("dev args examples:", len(examples))
text, loc, frame, fes = next(e for e in examples if e[3])  # one with FEs
trig = args_data.trigger_word_text(text, loc)
hint = args_data.frame_fe_hint(lex, frame)
combined, plen, ts, te = args_data.build_args_input(text, frame, loc, hint)
print("\nframe   :", frame)
print("trigger :", trig)
print("input   :", combined)  # trigger predicate-marked with <t> </t>
print("gold FEs:", [(name, text[s:e]) for name, s, e in fes])

## Train

Note: ~2·|FE vocab|+1 BIO labels (large head). `preprocess_logits_for_metrics`
keeps eval memory bounded. Expect a bit longer than the earlier slices.

In [ ]:
import train_args

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/texture_frames/args"
except Exception:
    OUTPUT_DIR = "outputs/args"
print("saving to:", OUTPUT_DIR)

model, tokenizer, lexicon = train_args.train(
    base_model="microsoft/deberta-v3-large",
    output_dir=OUTPUT_DIR,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=320,
)

## Evaluate — weighted span F1 (baseline-comparable)

In [ ]:
from eval_args import evaluate_args, print_report

metrics = evaluate_args(model, tokenizer, lexicon, split="test", max_length=320)
print_report(metrics, reported_f1=0.753)

## Interpreting

Report **args F1 vs 0.753**. This is the baseline's weakest task, so it's where
an encoder span head has the most room to win. If it underperforms, the most
likely lever is that the scaffold conveys the trigger only as a word in the
prefix (no explicit position marking) — adding a predicate-position marker is
the first Milestone 3 refinement for this head.